In [0]:
lst= dbutils.fs.ls("/Volumes/workspace/olist/volume_olist")
display(lst)

path,name,size,modificationTime
dbfs:/Volumes/workspace/olist/volume_olist/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1774936917000
dbfs:/Volumes/workspace/olist/volume_olist/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1774936926000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1774936916000
dbfs:/Volumes/workspace/olist/volume_olist/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1774936920000
dbfs:/Volumes/workspace/olist/volume_olist/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1774936914000
dbfs:/Volumes/workspace/olist/volume_olist/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1774936913000
dbfs:/Volumes/workspace/olist/volume_olist/product_category_name_translation.csv,product_category_name_translation.csv,2613,1774936913000


#### Olist_Customers

In [0]:
%sql 
SELECT *
FROM workspace.bronze.olist_customers
LIMIT 10

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG


##### Type Casting

In [0]:
%sql 
SELECT 
      *
FROM workspace.bronze.olist_customers
LIMIT 5

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [0]:

%sql
DESCRIBE workspace.bronze.olist_customers;

col_name,data_type,comment
customer_id,string,null
customer_unique_id,string,null
customer_zip_code_prefix,int,null
customer_city,string,null
customer_state,string,null


Only 1 field, customer_zip_code_prefix needs to be casted,despite it only consists of integer values, 
there are no need for arithmetic operations for zip code values, therefore it is better be string value

In [0]:
%sql
SELECT 
    customer_id
FROM workspace.bronze.olist_customers
WHERE customer_id != trim(customer_id)

customer_id


##### String Cleaning
- Unwanted Spaces
- Case correction

All columns are string, all fields will trimmed from white spaces and will be lower case 

In [0]:
%sql
SELECT 
      *
FROM workspace.bronze.olist_customers
LIMIT 5

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


##### NULL handling 

In [0]:
%sql 
SELECT 
    COUNT(*)
FROM workspace.bronze.olist_customers
WHERE customer_id IS NULL

COUNT(*)
0


In [0]:
%sql 
SELECT 
    COUNT(*) - COUNT(customer_id),
    COUNT(*) - COUNT(customer_unique_id),
    COUNT(*) - COUNT(customer_zip_code_prefix),
    COUNT(*) - COUNT(customer_city),
    COUNT(*) - COUNT(customer_state)
FROM workspace.bronze.olist_customers

COUNT(*)-COUNT(customer_id),COUNT(*)-COUNT(customer_unique_id),COUNT(*)-COUNT(customer_zip_code_prefix),COUNT(*)-COUNT(customer_city),COUNT(*)-COUNT(customer_state)
0,0,0,0,0


There is no null value 

##### Deduplication 

In [0]:
%sql 
SELECT
  customer_unique_id,
  COUNT(*)